# Cluster-Specific Rules

Owner: Mouaz

# Cluster-Specific Rules

Owner: Mouaz

We run Mohid's `mine_rules()` on the users of each of the 4 clusters from 02_clustering, and compare the rules between clusters.
A rule that is true for all users together may be stronger, weaker, or missing inside a specific audience group.

In [1]:
import sys
sys.path.append('..')

import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.rules import mine_rules
from src.data_utils import liked_ratings_with_genres

SEED = 42
DATA = '../data/'
FIG = '../figures/'

In [2]:
ratings = pd.read_parquet(DATA + 'clean_ratings.parquet')
movies = pd.read_parquet(DATA + 'movies_clean.parquet')
labels = pd.read_csv(DATA + 'cluster_labels.csv')

liked = liked_ratings_with_genres(ratings, movies)
del ratings   # free memory, we only need the liked ratings from now on

print('Liked ratings:', liked.shape)
print('Missing genres:', liked['genres'].isna().sum())
liked.head()

Liked ratings: (15914905, 4)
Missing genres: 0


,userId,movieId,rating,genres
0,1,17,4.0,Drama|Romance
3,1,30,5.0,Crime|Drama
4,1,32,5.0,Mystery|Sci-Fi|Thriller
7,1,80,5.0,Children|Drama
9,1,111,5.0,Crime|Drama|Thriller


In [3]:
MIN_SUPPORT, MIN_CONFIDENCE = 0.25, 0.85  

test_users = labels['userId'].sample(10000, random_state=SEED)
start = time.time()
test_rules = mine_rules(liked, test_users, MIN_SUPPORT, MIN_CONFIDENCE)
print(len(test_rules), 'rules found in', round(time.time() - start, 1), 's')
test_rules.head(10)

59 rules found in 4.8 s


,antecedents,consequents,support,confidence,lift
0,"Action, Sci-Fi",Adventure,0.293423,0.874925,1.519407
1,"Action, Drama, Sci-Fi",Adventure,0.263190,0.865657,1.503311
2,"Adventure, Sci-Fi",Action,0.293423,0.971173,1.424320
3,"Adventure, Drama, Sci-Fi",Action,0.263190,0.969038,1.421189
4,"Adventure, Thriller",Action,0.398338,0.946480,1.388105
5,"Sci-Fi, Thriller",Action,0.280108,0.945270,1.386332
6,"Drama, Sci-Fi, Thriller",Action,0.258184,0.943307,1.383452
7,"Adventure, Drama, Thriller",Action,0.372810,0.943262,1.383387
8,"Adventure, Thriller","Action, Drama",0.372810,0.885823,1.382360
9,"Adventure, Comedy, Thriller",Action,0.286215,0.939533,1.377918


In [4]:
from src.rules import build_transactions
from mlxtend.frequent_patterns import fpgrowth, association_rules

for t in [0.10, 0.20, 0.30]:
    tr = build_transactions(liked, test_users, share_threshold=t)
    freq = fpgrowth(tr, min_support=MIN_SUPPORT, use_colnames=True)
    rules = association_rules(freq, num_itemsets=len(tr), metric='confidence', min_threshold=MIN_CONFIDENCE)
    print(f'Share threshold {t:.2f}')
    print(f'  average genres per user: {tr.sum(axis=1).mean():.1f}')
    print(f'  rules: {len(rules)}, max lift: {rules["lift"].max() if len(rules) else 0:.2f}')
    print('  % of users with each genre:', (tr.mean() * 100).round(0).astype(int).sort_values(ascending=False).to_dict())
    print()

Share threshold 0.10
  average genres per user: 8.8
  rules: 3267, max lift: 1.27
  % of users with each genre: {'Drama': 99, 'Comedy': 95, 'Thriller': 91, 'Action': 88, 'Adventure': 85, 'Crime': 82, 'Romance': 74, 'Sci-Fi': 68, 'Fantasy': 50, 'Mystery': 38, 'Children': 31, 'Animation': 27, 'War': 22, 'Horror': 16, 'Musical': 10, 'Western': 2, 'Documentary': 1, 'Film-Noir': 1}

Share threshold 0.20
  average genres per user: 5.4
  rules: 59, max lift: 1.52
  % of users with each genre: {'Drama': 95, 'Comedy': 77, 'Thriller': 71, 'Action': 68, 'Adventure': 58, 'Crime': 47, 'Romance': 37, 'Sci-Fi': 36, 'Fantasy': 15, 'Children': 10, 'Animation': 8, 'Mystery': 6, 'War': 4, 'Horror': 4, 'Musical': 2, 'Film-Noir': 0, 'Documentary': 0, 'Western': 0}

Share threshold 0.30
  average genres per user: 3.2
  rules: 1, max lift: 1.00
  % of users with each genre: {'Drama': 86, 'Comedy': 50, 'Action': 44, 'Thriller': 41, 'Adventure': 32, 'Crime': 18, 'Romance': 15, 'Sci-Fi': 15, 'Fantasy': 4, 'Anim

### Choosing the share threshold for the transactions
A genre goes in a user's transaction if at least X% of the movies they liked have that genre. We tested 10%, 20% and 30% on 10,000 users (support 0.25, confidence 0.85):

| Share threshold | Genres per user | Rules | Best lift |
|---|---|---|---|
| 10% | 8.8 | 3,267 | 1.27 |
| 20% | 5.4 | 59 | 1.52 |
| 30% | 3.2 | 1 | 1.00 |

At 10%, almost every user has almost every genre (Drama 99%, Comedy 95%, Thriller 91%), so the rules are long, repetitive and weak. At 30%, only Drama is common enough to pass 25% support. **We use 20%**: a readable number of rules with clearly higher lift.

## Rules for all users and for each cluster
We run `mine_rules()` (support 0.25, confidence 0.85, share threshold 0.20) on all users, then separately on the users of each cluster.
Support is measured inside each group: a rule with support 0.30 in the Family cluster means 30% of that cluster's users have it.
We keep only rules with lift > 1.2, because Drama is in about 95% of transactions, so rules ending in Drama have lift close to 1 and say nothing beyond "Drama is popular".

In [5]:
CLUSTER_NAMES = {0: 'Drama & crime', 1: 'Comedy & romance',
                 2: 'Family & animation', 3: 'Action & sci-fi'}
MIN_LIFT = 1.2

groups = {'All users': labels['userId']}
for c, name in CLUSTER_NAMES.items():
    groups[name] = labels.loc[labels['cluster'] == c, 'userId']

group_rules = {}
for name, users in groups.items():
    start = time.time()
    r = mine_rules(liked, users, MIN_SUPPORT, MIN_CONFIDENCE)
    group_rules[name] = r[r['lift'] > MIN_LIFT].reset_index(drop=True)
    print(f'{name}: {len(users):,} users, {len(r)} rules, '
          f'{len(group_rules[name])} with lift > {MIN_LIFT} ({time.time() - start:.0f}s)')

All users: 200,948 users, 61 rules, 30 with lift > 1.2 (83s)
Drama & crime: 45,933 users, 71 rules, 6 with lift > 1.2 (15s)
Comedy & romance: 71,577 users, 42 rules, 0 with lift > 1.2 (29s)
Family & animation: 20,611 users, 232 rules, 25 with lift > 1.2 (6s)
Action & sci-fi: 62,827 users, 327 rules, 0 with lift > 1.2 (22s)


In [6]:
for name, r in group_rules.items():
    print(f'\n=== {name}: {len(r)} rules ===')
    if len(r):
        print(r.head(5).round(3).to_string(index=False))

rules_table = pd.concat([r.assign(group=name) for name, r in group_rules.items()],
                        ignore_index=True)
rules_table.to_csv(DATA + 'cluster_rules.csv', index=False)
print('\nSaved cluster_rules.csv:', rules_table.shape)


=== All users: 30 rules ===
             antecedents consequents  support  confidence  lift
          Action, Sci-Fi   Adventure    0.287       0.876 1.509
   Action, Drama, Sci-Fi   Adventure    0.261       0.868 1.495
       Adventure, Sci-Fi      Action    0.287       0.971 1.430
Adventure, Drama, Sci-Fi      Action    0.261       0.969 1.428
        Sci-Fi, Thriller      Action    0.274       0.949 1.397

=== Drama & crime: 6 rules ===
               antecedents   consequents  support  confidence  lift
       Adventure, Thriller Action, Drama    0.255       0.919 1.422
       Adventure, Thriller        Action    0.256       0.920 1.422
Adventure, Drama, Thriller        Action    0.255       0.920 1.422
                 Adventure Action, Drama    0.276       0.904 1.398
          Adventure, Drama        Action    0.276       0.905 1.398

=== Comedy & romance: 0 rules ===

=== Family & animation: 25 rules ===
                          antecedents         consequents  support  confid

## Comparing the same rule across groups
A rule can be missing from a group's list just because it fell below the support, confidence or lift threshold. To compare fairly, we take every rule found in any group and calculate its support, confidence and lift directly in every group.

In [7]:
from src.rules import build_transactions

trans = {}
for name, users in groups.items():
    trans[name] = build_transactions(liked, users, share_threshold=0.20)
    print(f'{name}: {trans[name].shape[0]:,} transactions')

All users: 200,726 transactions
Drama & crime: 45,922 transactions
Comedy & romance: 71,430 transactions
Family & animation: 20,592 transactions
Action & sci-fi: 62,782 transactions


In [8]:
def rule_stats(tr, ante, cons):
    """Support, confidence and lift of the rule ante -> cons inside one group's transactions."""
    has_a = tr.reindex(columns=ante, fill_value=False).all(axis=1)
    has_c = tr.reindex(columns=cons, fill_value=False).all(axis=1)
    support = (has_a & has_c).mean()
    confidence = support / has_a.mean() if has_a.mean() > 0 else np.nan
    lift = confidence / has_c.mean() if has_c.mean() > 0 else np.nan
    return support, confidence, lift

unique_rules = rules_table[['antecedents', 'consequents']].drop_duplicates()
rows = []
for _, rule in unique_rules.iterrows():
    ante, cons = rule['antecedents'].split(', '), rule['consequents'].split(', ')
    row = {'rule': f"{rule['antecedents']} -> {rule['consequents']}"}
    for name, tr in trans.items():
        s, c, l = rule_stats(tr, ante, cons)
        row[f'{name} lift'] = l
        row[f'{name} support'] = s
    rows.append(row)

compare = pd.DataFrame(rows).set_index('rule')
lift_table = compare[[c for c in compare.columns if c.endswith('lift')]]
support_table = compare[[c for c in compare.columns if c.endswith('support')]]

print(len(compare), 'unique rules')
lift_table.round(2)

51 unique rules


,All users lift,Drama & crime lift,Comedy & romance lift,Family & animation lift,Action & sci-fi lift
rule,,,,,
"Action, Sci-Fi -> Adventure",1.51,1.75,2.11,1.06,1.05
"Action, Drama, Sci-Fi -> Adventure",1.50,1.75,2.10,1.06,1.05
"Adventure, Sci-Fi -> Action",1.43,1.48,1.97,1.32,1.03
"Adventure, Drama, Sci-Fi -> Action",1.43,1.48,1.97,1.31,1.03
"Sci-Fi, Thriller -> Action",1.40,1.33,1.88,1.33,1.02
"Drama, Sci-Fi, Thriller -> Action",1.39,1.33,1.88,1.33,1.02
"Adventure, Thriller -> Action",1.39,1.42,1.94,1.25,1.02
"Adventure, Thriller -> Action, Drama",1.39,1.42,1.94,1.29,1.03
"Adventure, Drama, Thriller -> Action",1.39,1.42,1.94,1.24,1.02


In [9]:
support_table.round(2)

,All users support,Drama & crime support,Comedy & romance support,Family & animation support,Action & sci-fi support
rule,,,,,
"Action, Sci-Fi -> Adventure",0.29,0.11,0.06,0.37,0.65
"Action, Drama, Sci-Fi -> Adventure",0.26,0.11,0.06,0.30,0.59
"Adventure, Sci-Fi -> Action",0.29,0.11,0.06,0.37,0.65
"Adventure, Drama, Sci-Fi -> Action",0.26,0.11,0.06,0.30,0.59
"Sci-Fi, Thriller -> Action",0.27,0.19,0.05,0.16,0.62
"Drama, Sci-Fi, Thriller -> Action",0.25,0.19,0.05,0.13,0.57
"Adventure, Thriller -> Action",0.40,0.26,0.19,0.28,0.79
"Adventure, Thriller -> Action, Drama",0.38,0.26,0.19,0.24,0.73
"Adventure, Drama, Thriller -> Action",0.38,0.26,0.19,0.24,0.73


## Findings
| Rule (support / lift) | All users | Drama & crime | Comedy & romance | Family & animation | Action & sci-fi |
|---|---|---|---|---|---|
| Action, Sci-Fi -> Adventure | 29% / 1.51 | 11% / 1.75 | 6% / 2.11 | 37% / 1.06 | **65% / 1.05** |
| Crime -> Thriller | 41% / 1.26 | **77% / 1.04** | 22% / 1.44 | 8% / 1.93 | 48% / 1.05 |
| Animation, Comedy -> Children | 6% / 8.11 | 0% | 1% | **43% / 1.39** | 2% |
| Romance -> Comedy, Drama | 33% / 1.20 | 17% / 1.26 | **62% / 1.02** | 40% / 1.11 | 11% / 1.29 |

1. **In the cluster that matches a rule's genres, the rule is very common but its lift is close to 1**, because almost everyone in that group already likes all those genres.
2. **In other clusters the same rule is rare but has high lift**: it describes a small subgroup (e.g. 6% of Comedy & romance fans, lift 2.11).
3. **For all users, the lift is a mix of both.** Most rules found on all users mainly describe the audience groups, not a general link between genres.
4. **Animation, Comedy -> Children only appears inside the Family cluster** (43% support there, 6% overall). With a 25% minimum support it cannot be found without clustering.
5. **Lift is unreliable when support is tiny** (e.g. lift 80.9 with 0% support in Drama & crime), so lift is always read together with support.
6. 222 users never rated a movie 4.0 or higher, so they have no transaction (200,726 of 200,948).

In [ ]:
KEY_RULES = ['Action, Sci-Fi -> Adventure', 'Crime -> Thriller',
             'Animation, Comedy -> Children', 'Romance -> Comedy, Drama']
group_names = list(groups.keys())
colors = ['grey'] + ['tab:blue'] * len(CLUSTER_NAMES)

fig, axes = plt.subplots(1, 4, figsize=(17, 5), sharey=True)
for ax, rule in zip(axes, KEY_RULES):
    sup = [support_table.loc[rule, f'{g} support'] for g in group_names]
    lift = [lift_table.loc[rule, f'{g} lift'] for g in group_names]
    bars = ax.bar(range(len(group_names)), sup, color=colors)
    for bar, s, l in zip(bars, sup, lift):
        label = f'lift {l:.2f}' if s >= 0.05 else 'too rare'
        ax.text(bar.get_x() + bar.get_width() / 2, s + 0.01, label, ha='center', fontsize=8)
    ax.set_xticks(range(len(group_names)))
    ax.set_xticklabels(group_names, rotation=35, ha='right')
    ax.set_title(rule.replace(' -> ', ' → '), fontsize=11)
axes[0].set_ylabel('Support (share of users in the group)')
fig.suptitle('The same rule in each group: bar = how common it is, label = how strong it is (lift)')
plt.tight_layout()
plt.savefig(FIG + 'rules_across_clusters.png', dpi=150)
plt.show()